<a href="https://colab.research.google.com/github/mrivassnj-svg/HCC_ITAI_1371_SPR26/blob/main/PIG_IMPACT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
from google.colab import files
from sklearn.model_selection import train_test_split # Moved import here for early split

# ---------------------------
# 1. Automated Kaggle Setup
# ---------------------------
def setup_kaggle():
    !mkdir -p ~/.kaggle && echo KGAT_6d5e46e7d15065756f1bcd044124b6f8 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

setup_kaggle()

# ---------------------------
# 2. Download & Extract Datasets
# ---------------------------
datasets = [
    "shijo96john/animal-disease-prediction",
    "imdevskp/h1n1-swine-flu-2009-pandemic-dataset"
]

for ds in datasets:
    os.system(f'kaggle datasets download -d {ds}')
    zip_name = ds.split('/')[-1] + ".zip"
    os.system(f'unzip -o {zip_name}')

# ---------------------------
# 3. Process Dataset 1: Symptom Data
# ---------------------------
df_symptoms = pd.read_csv('cleaned_animal_disease_prediction.csv')
df_symptoms.columns = df_symptoms.columns.str.strip()

# Target Swine-related records
swine_symptoms = df_symptoms[
    df_symptoms['Animal_Type'].str.contains(r'pig|swine|hog', case=False, na=False)
].copy()

# ---------------------------
# 4. Process Dataset 2: Pandemic Trends (H1N1)
# ---------------------------
# This dataset provides the 'Global Context' of outbreaks
df_h1n1 = pd.read_csv('data.csv') # Changed from 'pandas_full_data.csv' to 'data.csv'

# We clean this to align with our symptom data structure
df_h1n1 = df_h1n1.rename(columns={
    'Country': 'Region',
    'Date': 'Date',
    'Cumulative no. of cases': 'case_count'
})
df_h1n1['disease_label'] = 'h1n1_swine_flu'

# ---------------------------
# 5. REVISED: Data Integration & Target Creation
# ---------------------------
def clean_pipeline(df):
    # Ensure Date column is datetime type
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

    # Standardize regions/labels
    region_col = 'Region' if 'Region' in df.columns else 'Geographical__Reported_Location'
    if region_col in df.columns:
        df['Region_Clean'] = df[region_col].astype(str).str.lower().str.strip()
    else:
        df['Region_Clean'] = None # Placeholder if no region found
    return df

swine_symptoms = clean_pipeline(swine_symptoms)
swine_trends = clean_pipeline(df_h1n1)

# Aggregate swine_trends to get global context features
# This replaces the problematic cross-join with meaningful summary features
h1n1_summary = swine_trends.agg({
    'case_count': ['max', 'mean'],
    'Cumulative no. of deaths': ['max', 'mean']
})
# Flatten the MultiIndex columns generated by .agg()
h1n1_summary.columns = ['_'.join(col).strip() for col in h1n1_summary.columns.values]

# Create master_df from swine_symptoms and add h1n1 summary (duplicate across all rows of swine_symptoms)
master_df = swine_symptoms.copy()

# Add the H1N1 global summary features to each row of master_df
# This applies the general H1N1 trend context to all swine records
for col in h1n1_summary.columns:
    master_df[col] = h1n1_summary.iloc[0][col]

# Convert Body_Temperature to numeric by stripping '°C' and Heart_Rate to numeric
if 'Body_Temperature' in master_df.columns:
    master_df['Body_Temperature'] = master_df['Body_Temperature'].astype(str).str.replace('°C', '', regex=False).astype(float)
if 'Heart_Rate' in master_df.columns:
    master_df['Heart_Rate'] = pd.to_numeric(master_df['Heart_Rate'], errors='coerce') # Coerce errors to NaN


# ---------------------------
# 6. Target Variable Creation (BEFORE splitting X and y)
# ---------------------------
binary_map = {'yes': 1, '+': 1, 'no': 0, '-': 0}
symptom_cols = ['Diarrhea', 'Coughing', 'Labored_Breathing'] # These will be dropped from X

for col in [col for col in symptom_cols if col in master_df.columns]:
    master_df[col] = master_df[col].astype(str).str.lower().map(binary_map).fillna(0)

# REVISED: Redefine target variable for binary classification
# Create the new binary target variable 'Is_Swine_Influenza'
# Where 1 indicates 'Swine Influenza' and 0 indicates any other disease
master_df['Is_Swine_Influenza'] = (master_df['Disease_Prediction'] == 'Swine Influenza').astype(int)

# ---------------------------
# 7. Define Features (X) and Target (y) & Initial Split (Early Split)
# ---------------------------
y = master_df['Is_Swine_Influenza'] # Use the newly defined target

# Define columns to drop from the feature set (X) to prevent data leakage and irrelevance
drop_from_X = [
    'Compounded_Complications', # Old target variable
    'Is_Swine_Influenza', # New target variable itself
    # Direct components of the target (leakage_cols) - removed from X to prevent leakage
    'Diarrhea', 'Coughing', 'Labored_Breathing',
    'patient_id', # Unique identifier, not a feature
    'Region', 'Date', # Original columns, 'Region_Clean' created from them is also dropped
    'Animal_Type', # Constant for 'swine' after filtering
    'Geographical__Reported_Location', # Redundant if 'Region_Clean' exists, or to be dropped
    'Region_Clean', # Dropping this from X as it's often sparse or complex for OHE if not handled carefully
    # Original h1n1 columns, replaced by aggregated summary features:
    'case_count',
    'Cumulative no. of deaths',
    'disease_label' # Constant value
]

# Also drop the original 'Disease_Prediction' column as it's now the basis for the target
if 'Disease_Prediction' in master_df.columns:
    drop_from_X.append('Disease_Prediction')

# Filter drop_from_X to only include columns actually present in master_df
drop_from_X = [col for col in drop_from_X if col in master_df.columns]

X = master_df.drop(columns=drop_from_X, errors='ignore')

# Ensure X does not contain any 'Link_' columns that may have been generated by accident
# These were previously identified as high-dimensional noise features (Issue 8)
link_cols = [col for col in X.columns if 'Link_' in col]
if link_cols:
    X = X.drop(columns=link_cols, errors='ignore')

# Perform train/validation/test split IMMEDIATELY after defining X and y
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# Store the initial y_train before SMOTE for plotting purposes
y_train_before_smote = y_train.copy()

print(f"✅ Data split into training, validation, and test sets. X_train shape: {X_train.shape}")
print(f"Initial X_train columns: {X_train.columns.tolist()}")
print("\nTarget variable (y) value counts:")
print(y.value_counts())

✅ Data split into training, validation, and test sets. X_train shape: (26, 19)
Initial X_train columns: ['Breed', 'Age', 'Gender', 'Weight', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Duration', 'Appetite_Loss', 'Vomiting', 'Lameness', 'Skin_Lesions', 'Nasal_Discharge', 'Eye_Discharge', 'Body_Temperature', 'Heart_Rate', 'c_a_s_e___c_o_u_n_t', 'C_u_m_u_l_a_t_i_v_e_ _n_o_._ _o_f_ _d_e_a_t_h_s']

Target variable (y) value counts:
Is_Swine_Influenza
0    28
1    10
Name: count, dtype: int64


In [2]:
from sklearn.preprocessing import OneHotEncoder

# Identify categorical and numerical columns based on X_train
categorical_cols = X_train.select_dtypes(include=['object', 'bool']).columns
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

# Handle potential high-cardinality nominal features like 'Region_Clean' or other 'Link_' columns
# Drop any columns that were implicitly created by string operations or are high-cardinality identifiers
# This addresses part of Issue 7 and 8 more explicitly for columns not caught by initial drop_from_X
link_cols_train = [col for col in categorical_cols if 'Link_' in col or col == 'Region_Clean']

if link_cols_train:
    X_train = X_train.drop(columns=link_cols_train)
    X_val = X_val.drop(columns=link_cols_train)
    X_test = X_test.drop(columns=link_cols_train)
    categorical_cols = [col for col in categorical_cols if col not in link_cols_train]

# One-Hot Encode categorical features
# Fit on X_train and transform all splits
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Transform training data
ohe.fit(X_train[categorical_cols])
X_train_ohe = ohe.transform(X_train[categorical_cols])
X_train_ohe_df = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(categorical_cols), index=X_train.index)

# Transform validation data
X_val_ohe = ohe.transform(X_val[categorical_cols])
X_val_ohe_df = pd.DataFrame(X_val_ohe, columns=ohe.get_feature_names_out(categorical_cols), index=X_val.index)

# Transform test data
X_test_ohe = ohe.transform(X_test[categorical_cols])
X_test_ohe_df = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(categorical_cols), index=X_test.index)

# Drop original categorical columns and concatenate one-hot encoded columns
X_train = pd.concat([X_train.drop(columns=categorical_cols), X_train_ohe_df], axis=1)
X_val = pd.concat([X_val.drop(columns=categorical_cols), X_val_ohe_df], axis=1)
X_test = pd.concat([X_test.drop(columns=categorical_cols), X_test_ohe_df], axis=1)

print(f"✅ Categorical features One-Hot Encoded. New X_train shape: {X_train.shape}")

✅ Categorical features One-Hot Encoded. New X_train shape: (26, 50)


In [3]:
import numpy as np

# Outlier Clipping for numerical features (Issue 6: Outlier Handling on Entire Dataset)
# Calculate Q1/Q3 only from X_train to prevent data leakage
for col in numerical_cols:
    if col in X_train.columns: # Check if column still exists after potential drops
        q1 = X_train[col].quantile(0.25)
        q3 = X_train[col].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        # Apply clipping to all splits using bounds from X_train
        X_train[col] = np.clip(X_train[col], lower_bound, upper_bound)
        X_val[col] = np.clip(X_val[col], lower_bound, upper_bound)
        X_test[col] = np.clip(X_test[col], lower_bound, upper_bound)

print(f"✅ Numerical features clipped for outliers using X_train statistics.")

✅ Numerical features clipped for outliers using X_train statistics.


In [4]:
# Ensure all splits have the same columns after encoding and dropping
# This addresses Issue 10: Model Evaluation on Inconsistent Feature Set
common_cols = list(set(X_train.columns) & set(X_val.columns) & set(X_test.columns))

X_train = X_train[common_cols]
X_val = X_val[common_cols]
X_test = X_test[common_cols]

print(f"✅ All feature sets synchronized to {len(common_cols)} common columns.")

✅ All feature sets synchronized to 50 common columns.


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Initialize models
models = {
    "Logistic": LogisticRegression(random_state=42, solver='liblinear'), # Added solver for robustness
    "Tree": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42),
    "GB": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(random_state=42, probability=True) # probability=True for ROC-AUC
}

results = {}

# Train all models on the training data
for name, model in models.items():
    print(f"Training {name} model...")
    model.fit(X_train, y_train)
    results[name] = model
print("✅ All individual models trained.")

Training Logistic model...
Training Tree model...
Training RF model...
Training GB model...
Training KNN model...
Training SVC model...
✅ All individual models trained.


Step 4 Validate Metrics (Table Needs to Be Inserted)

In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

validation_metrics = {}

# Evaluate all models on the validation set
for name, model in results.items():
    preds = model.predict(X_val)

    # For ROC-AUC, we need probabilities for binary classification
    # Check if the model has predict_proba, otherwise use decision_function or default to 0.5 for binary
    if hasattr(model, 'predict_proba'):
        preds_proba = model.predict_proba(X_val)[:, 1]
    else:
        # SVC without probability=True does not have predict_proba by default
        # For models like SVC without probability=True, ROC-AUC cannot be calculated directly
        preds_proba = (preds > 0.5).astype(int) # This is a fallback, not ideal for ROC-AUC

    validation_metrics[name] = {
        "Accuracy": accuracy_score(y_val, preds),
        "Precision": precision_score(y_val, preds, zero_division=0),
        "Recall": recall_score(y_val, preds, zero_division=0),
        "F1-Score": f1_score(y_val, preds, zero_division=0)
    }
    try:
        validation_metrics[name]["ROC-AUC"] = roc_auc_score(y_val, preds_proba)
    except ValueError: # Handle cases where only one class is present in y_true, making AUC undefined
        validation_metrics[name]["ROC-AUC"] = None # Or np.nan

# Convert validation metrics to DataFrame for easier analysis
val_df = pd.DataFrame(validation_metrics).T.reset_index()
val_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

print("✅ Validation metrics calculated:")
display(val_df)

✅ Validation metrics calculated:


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic,0.666667,0.0,0.0,0.0,0.0
1,Tree,0.333333,0.0,0.0,0.0,0.2
2,RF,0.500000,0.0,0.0,0.0,0.0
3,GB,0.333333,0.0,0.0,0.0,0.0
4,KNN,0.833333,0.0,0.0,0.0,0.0
5,SVC,0.833333,0.0,0.0,0.0,0.6


Step 5 Ensemble Model

In [7]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Dynamically pick best models for the ensemble based on validation ROC-AUC score
# Sort models by ROC-AUC score in descending order and select the top N
num_ensemble_models = 3 # Can be adjusted

# Filter out models with None for ROC-AUC before sorting, if any
val_df_filtered = val_df[val_df['ROC-AUC'].notna()].copy()

if not val_df_filtered.empty:
    best_models_for_ensemble = val_df_filtered.sort_values(by='ROC-AUC', ascending=False).head(num_ensemble_models)['Model'].tolist()
else:
    print("Warning: No models with valid ROC-AUC scores found for ensemble selection. Defaulting to all models.")
    best_models_for_ensemble = val_df['Model'].tolist() # Fallback to all models if no valid ROC-AUC

print(f"Dynamically selected models for ensemble: {best_models_for_ensemble}")

def ensemble_predict(X_data, selected_models, threshold=0.5):
    # Ensure models exist in the 'results' dictionary
    preds_array = []
    for m_name in selected_models:
        if m_name in results:
            model = results[m_name]
            if hasattr(model, 'predict_proba'):
                # Use probabilities for a soft voting ensemble, then threshold
                preds_array.append(model.predict_proba(X_data)[:, 1])
            else:
                # Fallback for models without predict_proba (like SVC without probability=True)
                # For models with hard predictions only, we would directly average the predicted classes
                preds_array.append(model.predict(X_data))
        else:
            print(f"Warning: Model {m_name} not found for ensemble.")

    if preds_array:
        # Average the probabilities (soft voting) or raw predictions (hard voting for discrete output)
        # For soft voting, we average probabilities and then apply a threshold
        avg_preds = np.mean(np.column_stack(preds_array), axis=1)
        return (avg_preds >= threshold).astype(int)
    else:
        return np.array([])

# Get validation predictions from the ensemble
val_preds_ensemble = ensemble_predict(X_val, best_models_for_ensemble)

# Calculate and store ensemble validation metrics
if val_preds_ensemble.size > 0:
    ensemble_val_accuracy = accuracy_score(y_val, val_preds_ensemble)
    ensemble_val_precision = precision_score(y_val, val_preds_ensemble, zero_division=0)
    ensemble_val_recall = recall_score(y_val, val_preds_ensemble, zero_division=0)
    ensemble_val_f1 = f1_score(y_val, val_preds_ensemble, zero_division=0)

    # ROC-AUC needs probabilities, re-run ensemble_predict for probabilities
    # Only include models with predict_proba for ROC-AUC calculation
    proba_models = [m_name for m_name in best_models_for_ensemble if m_name in results and hasattr(results[m_name], 'predict_proba')]
    if proba_models:
        ensemble_val_proba = np.mean(np.column_stack([results[m_name].predict_proba(X_val)[:, 1] for m_name in proba_models]), axis=1)
        try:
            ensemble_val_roc_auc = roc_auc_score(y_val, ensemble_val_proba)
        except ValueError:
            ensemble_val_roc_auc = None # Handle cases where only one class is present
    else:
        ensemble_val_roc_auc = None # No models with predict_proba found

    # Add ensemble metrics to validation_metrics dictionary
    validation_metrics['Ensemble'] = {
        "Accuracy": ensemble_val_accuracy,
        "Precision": ensemble_val_precision,
        "Recall": ensemble_val_recall,
        "F1-Score": ensemble_val_f1,
        "ROC-AUC": ensemble_val_roc_auc
    }

    # Recreate val_df to include ensemble metrics
    val_df = pd.DataFrame(validation_metrics).T.reset_index()
    val_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

    print("✅ Ensemble model validated.")
else:
    print("❌ Ensemble validation skipped due to no models selected or found.")

Dynamically selected models for ensemble: ['SVC', 'Tree', 'Logistic']
✅ Ensemble model validated.


Step 6 Test Evaluation

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Calculate ensemble predictions on the test set
test_preds_ensemble = ensemble_predict(X_test, best_models_for_ensemble)

# Calculate ensemble test metrics
if test_preds_ensemble.size > 0:
    ensemble_test_accuracy = accuracy_score(y_test, test_preds_ensemble)
    ensemble_test_precision = precision_score(y_test, test_preds_ensemble, zero_division=0)
    ensemble_test_recall = recall_score(y_test, test_preds_ensemble, zero_division=0)
    ensemble_test_f1 = f1_score(y_test, test_preds_ensemble, zero_division=0)

    # ROC-AUC needs probabilities, re-run ensemble_predict for probabilities
    # Only include models with predict_proba for ROC-AUC calculation
    proba_models = [m_name for m_name in best_models_for_ensemble if m_name in results and hasattr(results[m_name], 'predict_proba')]
    if proba_models:
        ensemble_test_proba = np.mean(np.column_stack([results[m_name].predict_proba(X_test)[:, 1] for m_name in proba_models]), axis=1)
        try:
            ensemble_test_roc_auc = roc_auc_score(y_test, ensemble_test_proba)
        except ValueError:
            ensemble_test_roc_auc = None # Handle cases where only one class is present
    else:
        ensemble_test_roc_auc = None # No models with predict_proba found

    print("✅ Ensemble Test Metrics:")
    print(f"  Test Accuracy: {ensemble_test_accuracy:.6f}")
    print(f"  Test Precision: {ensemble_test_precision:.6f}")
    print(f"  Test Recall: {ensemble_test_recall:.6f}")
    print(f"  Test F1-Score: {ensemble_test_f1:.6f}")
    if ensemble_test_roc_auc is not None:
        print(f"  Test ROC-AUC: {ensemble_test_roc_auc:.6f}")
    else:
        print("  Test ROC-AUC: Not calculable (single class in true labels)")
else:
    print("❌ Ensemble test evaluation skipped due to no models selected or found.")

✅ Ensemble Test Metrics:
  Test Accuracy: 0.666667
  Test Precision: 0.000000
  Test Recall: 0.000000
  Test F1-Score: 0.000000
  Test ROC-AUC: nan


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Step 7 Bayesian Model(Compare w Ensemble)

In [9]:
from sklearn.naive_bayes import GaussianNB # Changed to Gaussian Naive Bayes for classification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Training Bayesian (Gaussian Naive Bayes) model...")
bayes = GaussianNB() # Initialize Gaussian Naive Bayes classifier
bayes.fit(X_train, y_train)

# Predict on validation set and calculate metrics
preds_bayes_val = bayes.predict(X_val)

bayes_val_accuracy = accuracy_score(y_val, preds_bayes_val)
bayes_val_precision = precision_score(y_val, preds_bayes_val, zero_division=0)
bayes_val_recall = recall_score(y_val, preds_bayes_val, zero_division=0)
bayes_val_f1 = f1_score(y_val, preds_bayes_val, zero_division=0)

# For ROC-AUC, we need probabilities
if hasattr(bayes, 'predict_proba'):
    preds_bayes_val_proba = bayes.predict_proba(X_val)[:, 1]
    try:
        bayes_val_roc_auc = roc_auc_score(y_val, preds_bayes_val_proba)
    except ValueError:
        bayes_val_roc_auc = None
else:
    bayes_val_roc_auc = None

# Add Bayesian metrics to validation_metrics dictionary and val_df
validation_metrics['Bayesian'] = {
    "Accuracy": bayes_val_accuracy,
    "Precision": bayes_val_precision,
    "Recall": bayes_val_recall,
    "F1-Score": bayes_val_f1,
    "ROC-AUC": bayes_val_roc_auc
}

# Recreate val_df to include Bayesian metrics
val_df = pd.DataFrame(validation_metrics).T.reset_index()
val_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

print("✅ Bayesian (Gaussian Naive Bayes) model trained and validated.")

Training Bayesian (Gaussian Naive Bayes) model...
✅ Bayesian (Gaussian Naive Bayes) model trained and validated.


Step 8 Deliverable Create Comparison Table

In [10]:
# Convert validation metrics to DataFrame
# This val_df is already updated by previous cells
print("Validation Metrics:")
display(val_df)

Validation Metrics:


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic,0.666667,0.0,0.0,0.000000,0.0
1,Tree,0.333333,0.0,0.0,0.000000,0.2
2,RF,0.500000,0.0,0.0,0.000000,0.0
3,GB,0.333333,0.0,0.0,0.000000,0.0
4,KNN,0.833333,0.0,0.0,0.000000,0.0
5,SVC,0.833333,0.0,0.0,0.000000,0.6
6,Ensemble,0.333333,0.0,0.0,0.000000,0.0
7,Bayesian,0.333333,0.2,1.0,0.333333,0.2


Step 9 Evaluate All Models

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

test_metrics = {}

# Evaluate all individual models on the test set
for name, model in results.items():
    preds = model.predict(X_test)

    # For ROC-AUC, we need probabilities for binary classification
    if hasattr(model, 'predict_proba'):
        preds_proba = model.predict_proba(X_test)[:, 1]
    else:
        preds_proba = (preds > 0.5).astype(int) # Fallback

    test_metrics[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1-Score": f1_score(y_test, preds, zero_division=0)
    }
    try:
        test_metrics[name]["ROC-AUC"] = roc_auc_score(y_test, preds_proba)
    except ValueError:
        test_metrics[name]["ROC-AUC"] = None # Handle cases where only one class is present

# Convert test metrics to DataFrame
test_df = pd.DataFrame(test_metrics).T.reset_index()
test_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

print("✅ Individual Model Test Metrics:")
display(test_df)

✅ Individual Model Test Metrics:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic,0.500000,0.0,0.0,0.0,NaN
1,Tree,0.666667,0.0,0.0,0.0,NaN
2,RF,0.666667,0.0,0.0,0.0,NaN
3,GB,0.500000,0.0,0.0,0.0,NaN
4,KNN,1.000000,0.0,0.0,0.0,NaN
5,SVC,1.000000,0.0,0.0,0.0,NaN


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Ensemble Test Metrics (from previous cell)
# These variables (ensemble_test_accuracy, etc.) should already be defined from cell -D__J-47pfL0
ensemble_test = {
    "Accuracy": ensemble_test_accuracy,
    "Precision": ensemble_test_precision,
    "Recall": ensemble_test_recall,
    "F1-Score": ensemble_test_f1,
    "ROC-AUC": ensemble_test_roc_auc
}

# Bayesian Test Metrics
bayes_preds_test = bayes.predict(X_test)

# For ROC-AUC, we need probabilities
if hasattr(bayes, 'predict_proba'):
    preds_bayes_test_proba = bayes.predict_proba(X_test)[:, 1]
    try:
        bayes_test_roc_auc = roc_auc_score(y_test, preds_bayes_test_proba)
    except ValueError:
        bayes_test_roc_auc = None
else:
    bayes_test_roc_auc = None

bayes_test = {
    "Accuracy": accuracy_score(y_test, bayes_preds_test),
    "Precision": precision_score(y_test, bayes_preds_test, zero_division=0),
    "Recall": recall_score(y_test, bayes_preds_test, zero_division=0),
    "F1-Score": f1_score(y_test, bayes_preds_test, zero_division=0),
    "ROC-AUC": bayes_test_roc_auc
}

# Append Ensemble and Bayesian results to test_df
extra_models = pd.DataFrame([
    ["Ensemble", ensemble_test["Accuracy"], ensemble_test["Precision"], ensemble_test["Recall"], ensemble_test["F1-Score"], ensemble_test["ROC-AUC"]],
    ["Bayesian", bayes_test["Accuracy"], bayes_test["Precision"], bayes_test["Recall"], bayes_test["F1-Score"], bayes_test["ROC-AUC"]]
], columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])

test_df = pd.concat([test_df, extra_models], ignore_index=True)

print("✅ All Model Test Metrics (Individual, Ensemble, Bayesian):")
display(test_df)

In [12]:
# Ensemble Test Metrics (from previous cell)
ensemble_test = {
    "MAE": ensemble_test_mae,
    "MSE": ensemble_test_mse,
    "R2": ensemble_test_r2
}

# Bayesian Test Metrics
bayes_preds_test = bayes.predict(X_test)

bayes_test = {
    "MAE": mean_absolute_error(y_test, bayes_preds_test),
    "MSE": mean_squared_error(y_test, bayes_preds_test),
    "R2": r2_score(y_test, bayes_preds_test)
}

# Append Ensemble and Bayesian results to test_df
extra_models = pd.DataFrame([
    ["Ensemble", ensemble_test["MAE"], ensemble_test["MSE"], ensemble_test["R2"]],
    ["Bayesian", bayes_test["MAE"], bayes_test["MSE"], bayes_test["R2"]]
], columns=['Model', 'MAE', 'MSE', 'R2'])

test_df = pd.concat([test_df, extra_models], ignore_index=True)

print("✅ All Model Test Metrics (Individual, Ensemble, Bayesian):")
display(test_df)

NameError: name 'ensemble_test_mae' is not defined

Step 11 Save Outputs

In [ ]:
# Save tables
val_df.to_csv("validation_metrics.csv", index=False)
test_df.to_csv("test_metrics.csv", index=False)

# Save final dataset
master_df.to_csv("final_processed_dataset.csv", index=False)

print("✅ Files saved for submission")

Step 12 Auto-Select Best Model

In [ ]:
# Select the best model based on an appropriate classification metric on the test set (e.g., F1-Score or Accuracy)
# Given the issue of a single class in y_test making ROC-AUC and other metrics undefined for the minority class,
# we might choose Accuracy or F1-Score with zero_division=0, or acknowledge the limitations.
# For now, let's prioritize F1-Score (as it balances precision and recall), handling NaNs if they appear for ROC-AUC.

# Filter out models where F1-Score is NaN (e.g., due to single-class y_true combined with zero_division=0 for positive class)
# If all F1-Scores are 0.0, it will sort by Accuracy next.

# First, try to sort by 'F1-Score', then 'Accuracy' as a fallback for ties or cases where F1 might be universally low/zero
if 'F1-Score' in test_df.columns:
    # Fill None/NaN ROC-AUC for sorting purposes, e.g., with -1 to push them to the bottom
    test_df['ROC-AUC_filled'] = test_df['ROC-AUC'].fillna(-1)
    best_model_test = test_df.sort_values(by=['F1-Score', 'Accuracy', 'ROC-AUC_filled'], ascending=[False, False, False]).iloc[0]
    # Drop the temporary column before printing
    test_df = test_df.drop(columns=['ROC-AUC_filled'])
else:
    # Fallback if F1-Score column isn't available for some reason (shouldn't happen now)
    best_model_test = test_df.sort_values(by='Accuracy', ascending=False).iloc[0]

print("✅ Best Model on Test Set (based on F1-Score, then Accuracy, then ROC-AUC):")
print(best_model_test)

### Investigating the Target Variable: `Compounded_Complications`

### Class Distribution Before SMOTE

Before applying any resampling techniques, it's crucial to visualize the class distribution of our target variable (`Is_Swine_Influenza`) in the training set. This helps confirm the presence and severity of class imbalance, which can negatively impact model training.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Visualize class distribution before SMOTE
plt.figure(figsize=(7, 5))
sns.countplot(x=y_train_before_smote, palette='viridis') # Use y_train_before_smote
plt.title('Class Distribution of Is_Swine_Influenza (Before SMOTE)')
plt.xlabel('Class (0: Other, 1: Swine Influenza)')
plt.ylabel('Number of Samples')
plt.show()

print(f"Class distribution before SMOTE: {Counter(y_train_before_smote)}") # Use y_train_before_smote

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Describe the target variable
display(y.describe())

# Value counts of the target variable
display(y.value_counts().sort_index())

# Visualize the distribution of the target variable
plt.figure(figsize=(8, 5))
sns.countplot(x=y)
plt.title('Distribution of Compounded_Complications (Target Variable)')
plt.xlabel('Compounded Complications Count (0 to 3)')
plt.ylabel('Number of Swine Records')
plt.show()

print("\nObservations on target variable distribution:")
print("1. The target variable `Compounded_Complications` ranges from 0 to 3, representing the sum of 3 binary symptoms.")
print("2. It has very few discrete values, making it behave more like an ordinal or categorical variable rather than a continuous one suitable for typical regression.")
print("3. The distribution is skewed, with most values being 2 or 3. This sparsity and limited range likely contribute to the negative R-squared values, as traditional regression models struggle to find continuous relationships.")
print("4. For such a target, a classification approach (e.g., predicting severity levels or presence/absence of any complication) might be more appropriate, or a specialized regression model for count data.")

### Addressing Class Imbalance with SMOTE

As noted in the analysis, there's a significant class imbalance in the target variable `Is_Swine_Influenza`. This can lead models to perform poorly on the minority class. SMOTE (Synthetic Minority Over-sampling Technique) is a popular technique to address this by generating synthetic samples from the minority class.

In [ ]:
# Install imbalanced-learn library if not already installed
!pip install imbalanced-learn

import pandas as pd
from imblearn.over_sampling import SMOTE
from collections import Counter

print(f"Original training set shape: {X_train.shape}, labels: {y_train.shape}")
print(f"Original class distribution: {Counter(y_train)}")

# Apply SMOTE to the training data only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled training set shape: {X_train_resampled.shape}, labels: {y_train_resampled.shape}")
print(f"Resampled class distribution: {Counter(y_train_resampled)}")

# Update X_train and y_train to the resampled versions
X_train = X_train_resampled
y_train = y_train_resampled

print("✅ Training data resampled using SMOTE.")

### Class Distribution After SMOTE

After applying SMOTE to the training data, we can re-examine the class distribution to confirm that the minority class has been sufficiently oversampled, leading to a more balanced dataset for model training.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Visualize class distribution after SMOTE (using the updated y_train)
plt.figure(figsize=(7, 5))
sns.countplot(x=y_train, palette='viridis')
plt.title('Class Distribution of Is_Swine_Influenza (After SMOTE)')
plt.xlabel('Class (0: Other, 1: Swine Influenza)')
plt.ylabel('Number of Samples')
plt.show()

print(f"Class distribution after SMOTE: {Counter(y_train)}")

Now that the training data (`X_train` and `y_train`) has been resampled using SMOTE, the class distribution is balanced. You can now re-train your models using these balanced datasets to see if it improves their performance, especially concerning the prediction of the minority class ('Swine Influenza').

### Re-training Models with Resampled Data

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Initialize models (re-initialize to ensure fresh training)
models = {
    "Logistic": LogisticRegression(random_state=42, solver='liblinear'),
    "Tree": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42),
    "GB": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(random_state=42, probability=True)
}

results = {}

# Train all models on the RESAMPLED training data
for name, model in models.items():
    print(f"Training {name} model with resampled data...")
    model.fit(X_train, y_train)
    results[name] = model
print("✅ All individual models re-trained with resampled data.")

### Re-validating Models

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

validation_metrics = {}

# Evaluate all models on the validation set
for name, model in results.items():
    preds = model.predict(X_val)

    if hasattr(model, 'predict_proba'):
        preds_proba = model.predict_proba(X_val)[:, 1]
    else:
        preds_proba = (preds > 0.5).astype(int)

    validation_metrics[name] = {
        "Accuracy": accuracy_score(y_val, preds),
        "Precision": precision_score(y_val, preds, zero_division=0),
        "Recall": recall_score(y_val, preds, zero_division=0),
        "F1-Score": f1_score(y_val, preds, zero_division=0)
    }
    try:
        validation_metrics[name]["ROC-AUC"] = roc_auc_score(y_val, preds_proba)
    except ValueError:
        validation_metrics[name]["ROC-AUC"] = None

# Convert validation metrics to DataFrame for easier analysis
val_df = pd.DataFrame(validation_metrics).T.reset_index()
val_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

print("✅ Validation metrics re-calculated:")
display(val_df)

### Re-evaluating Ensemble Model

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Dynamically pick best models for the ensemble based on validation ROC-AUC score
num_ensemble_models = 3
val_df_filtered = val_df[val_df['ROC-AUC'].notna()].copy()

if not val_df_filtered.empty:
    best_models_for_ensemble = val_df_filtered.sort_values(by='ROC-AUC', ascending=False).head(num_ensemble_models)['Model'].tolist()
else:
    print("Warning: No models with valid ROC-AUC scores found for ensemble selection. Defaulting to all models.")
    best_models_for_ensemble = val_df['Model'].tolist()

print(f"Dynamically selected models for ensemble: {best_models_for_ensemble}")

def ensemble_predict(X_data, selected_models, threshold=0.5):
    preds_array = []
    for m_name in selected_models:
        if m_name in results:
            model = results[m_name]
            if hasattr(model, 'predict_proba'):
                preds_array.append(model.predict_proba(X_data)[:, 1])
            else:
                preds_array.append(model.predict(X_data))
        else:
            print(f"Warning: Model {m_name} not found for ensemble.")

    if preds_array:
        avg_preds = np.mean(np.column_stack(preds_array), axis=1)
        return (avg_preds >= threshold).astype(int)
    else:
        return np.array([])

# Get validation predictions from the ensemble
val_preds_ensemble = ensemble_predict(X_val, best_models_for_ensemble)

# Calculate and store ensemble validation metrics
if val_preds_ensemble.size > 0:
    ensemble_val_accuracy = accuracy_score(y_val, val_preds_ensemble)
    ensemble_val_precision = precision_score(y_val, val_preds_ensemble, zero_division=0)
    ensemble_val_recall = recall_score(y_val, val_preds_ensemble, zero_division=0)
    ensemble_val_f1 = f1_score(y_val, val_preds_ensemble, zero_division=0)

    proba_models = [m_name for m_name in best_models_for_ensemble if m_name in results and hasattr(results[m_name], 'predict_proba')]
    if proba_models:
        ensemble_val_proba = np.mean(np.column_stack([results[m_name].predict_proba(X_val)[:, 1] for m_name in proba_models]), axis=1)
        try:
            ensemble_val_roc_auc = roc_auc_score(y_val, ensemble_val_proba)
        except ValueError:
            ensemble_val_roc_auc = None
    else:
        ensemble_val_roc_auc = None

    validation_metrics['Ensemble'] = {
        "Accuracy": ensemble_val_accuracy,
        "Precision": ensemble_val_precision,
        "Recall": ensemble_val_recall,
        "F1-Score": ensemble_val_f1,
        "ROC-AUC": ensemble_val_roc_auc
    }

    val_df = pd.DataFrame(validation_metrics).T.reset_index()
    val_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

    print("✅ Ensemble model re-validated.")
else:
    print("❌ Ensemble validation skipped due to no models selected or found.")

### Re-evaluating Bayesian Model

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Training Bayesian (Gaussian Naive Bayes) model with resampled data...")
bayes = GaussianNB()
bayes.fit(X_train, y_train)

preds_bayes_val = bayes.predict(X_val)

bayes_val_accuracy = accuracy_score(y_val, preds_bayes_val)
bayes_val_precision = precision_score(y_val, preds_bayes_val, zero_division=0)
bayes_val_recall = recall_score(y_val, preds_bayes_val, zero_division=0)
bayes_val_f1 = f1_score(y_val, preds_bayes_val, zero_division=0)

if hasattr(bayes, 'predict_proba'):
    preds_bayes_val_proba = bayes.predict_proba(X_val)[:, 1]
    try:
        bayes_val_roc_auc = roc_auc_score(y_val, preds_bayes_val_proba)
    except ValueError:
        bayes_val_roc_auc = None
else:
    bayes_val_roc_auc = None

validation_metrics['Bayesian'] = {
    "Accuracy": bayes_val_accuracy,
    "Precision": bayes_val_precision,
    "Recall": bayes_val_recall,
    "F1-Score": bayes_val_f1,
    "ROC-AUC": bayes_val_roc_auc
}

val_df = pd.DataFrame(validation_metrics).T.reset_index()
val_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

print("✅ Bayesian (Gaussian Naive Bayes) model re-trained and re-validated.")

### Updated Validation Metrics Table

In [ ]:
print("Validation Metrics (after SMOTE):")
display(val_df)

### Re-evaluating All Models on Test Set

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

test_metrics = {}

# Evaluate all individual models on the test set
for name, model in results.items():
    preds = model.predict(X_test)

    if hasattr(model, 'predict_proba'):
        preds_proba = model.predict_proba(X_test)[:, 1]
    else:
        preds_proba = (preds > 0.5).astype(int)

    test_metrics[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1-Score": f1_score(y_test, preds, zero_division=0)
    }
    try:
        test_metrics[name]["ROC-AUC"] = roc_auc_score(y_test, preds_proba)
    except ValueError:
        test_metrics[name]["ROC-AUC"] = None

# Convert test metrics to DataFrame
test_df = pd.DataFrame(test_metrics).T.reset_index()
test_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

print("✅ Individual Model Test Metrics (after SMOTE):")
display(test_df)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Ensemble Test Metrics
test_preds_ensemble = ensemble_predict(X_test, best_models_for_ensemble)

if test_preds_ensemble.size > 0:
    ensemble_test_accuracy = accuracy_score(y_test, test_preds_ensemble)
    ensemble_test_precision = precision_score(y_test, test_preds_ensemble, zero_division=0)
    ensemble_test_recall = recall_score(y_test, test_preds_ensemble, zero_division=0)
    ensemble_test_f1 = f1_score(y_test, test_preds_ensemble, zero_division=0)

    proba_models = [m_name for m_name in best_models_for_ensemble if m_name in results and hasattr(results[m_name], 'predict_proba')]
    if proba_models:
        ensemble_test_proba = np.mean(np.column_stack([results[m_name].predict_proba(X_test)[:, 1] for m_name in proba_models]), axis=1)
        try:
            ensemble_test_roc_auc = roc_auc_score(y_test, ensemble_test_proba)
        except ValueError:
            ensemble_test_roc_auc = None
    else:
        ensemble_test_roc_auc = None

    ensemble_test_results = {
        "Accuracy": ensemble_test_accuracy,
        "Precision": ensemble_test_precision,
        "Recall": ensemble_test_recall,
        "F1-Score": ensemble_test_f1,
        "ROC-AUC": ensemble_test_roc_auc
    }
else:
    ensemble_test_results = {"Accuracy": None, "Precision": None, "Recall": None, "F1-Score": None, "ROC-AUC": None}

# Bayesian Test Metrics
bayes_preds_test = bayes.predict(X_test)

if hasattr(bayes, 'predict_proba'):
    preds_bayes_test_proba = bayes.predict_proba(X_test)[:, 1]
    try:
        bayes_test_roc_auc = roc_auc_score(y_test, preds_bayes_test_proba)
    except ValueError:
        bayes_test_roc_auc = None
else:
    bayes_test_roc_auc = None

bayes_test_results = {
    "Accuracy": accuracy_score(y_test, bayes_preds_test),
    "Precision": precision_score(y_test, bayes_preds_test, zero_division=0),
    "Recall": recall_score(y_test, bayes_preds_test, zero_division=0),
    "F1-Score": f1_score(y_test, bayes_preds_test, zero_division=0),
    "ROC-AUC": bayes_test_roc_auc
}

# Append Ensemble and Bayesian results to test_df
extra_models = pd.DataFrame([
    ["Ensemble", ensemble_test_results["Accuracy"], ensemble_test_results["Precision"], ensemble_test_results["Recall"], ensemble_test_results["F1-Score"], ensemble_test_results["ROC-AUC"]],
    ["Bayesian", bayes_test_results["Accuracy"], bayes_test_results["Precision"], bayes_test_results["Recall"], bayes_test_results["F1-Score"], bayes_test_results["ROC-AUC"]]
], columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])

test_df = pd.concat([test_df, extra_models], ignore_index=True)

print("✅ All Model Test Metrics (Individual, Ensemble, Bayesian) (after SMOTE):")
display(test_df)

### Auto-Select Best Model (After SMOTE)

In [ ]:
# Select the best model based on an appropriate classification metric on the test set (e.g., F1-Score or Accuracy)
# Given the issue of a single class in y_test making ROC-AUC and other metrics undefined for the minority class,
# we might choose Accuracy or F1-Score with zero_division=0, or acknowledge the limitations.
# For now, let's prioritize F1-Score (as it balances precision and recall), handling NaNs if they appear for ROC-AUC.

# Filter out models where F1-Score is NaN (e.g., due to single-class y_true combined with zero_division=0 for positive class)
# If all F1-Scores are 0.0, it will sort by Accuracy next.

# First, try to sort by 'F1-Score', then 'Accuracy' as a fallback for ties or cases where F1 might be universally low/zero
if 'F1-Score' in test_df.columns:
    # Fill None/NaN ROC-AUC for sorting purposes, e.g., with -1 to push them to the bottom
    test_df['ROC-AUC_filled'] = test_df['ROC-AUC'].fillna(-1)
    best_model_test = test_df.sort_values(by=['F1-Score', 'Accuracy', 'ROC-AUC_filled'], ascending=[False, False, False]).iloc[0]
    # Drop the temporary column before printing
    test_df = test_df.drop(columns=['ROC-AUC_filled'])
else:
    # Fallback if F1-Score column isn't available for some reason (shouldn't happen now)
    best_model_test = test_df.sort_values(by='Accuracy', ascending=False).iloc[0]

print("✅ Best Model on Test Set (based on F1-Score, then Accuracy, then ROC-AUC) (after SMOTE):")
print(best_model_test)

### Re-generating Final Report (After SMOTE)

In [ ]:
# ---------------------------
# Install Libraries
# ---------------------------
!pip install reportlab matplotlib

# ---------------------------
# Imports
# ---------------------------
import matplotlib.pyplot as plt
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.styles import getSampleStyleSheet
import datetime

# ---------------------------
# 1. Create Overall Model Comparison Graph
# ---------------------------
plt.figure(figsize=(10,5))
plt.bar(test_df['Model'], test_df['F1-Score'])
plt.title("Model Comparison (F1-Score) (After SMOTE)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("model_comparison_smote.png") # Save with a new name
plt.close()

# ---------------------------
# 2. Create Individual Model Plots
# ---------------------------
model_images_smote = {} # New dictionary for SMOTE plots

for name, model in results.items():
    preds = model.predict(X_test)

    plt.figure()
    plt.scatter(y_test, preds, alpha=0.5)
    plt.xlabel("Actual (Is_Swine_Influenza)")
    plt.ylabel("Predicted (Is_Swine_Influenza)")
    plt.title(f"{name} Model Predictions (After SMOTE)")

    filename = f"{name}_plot_smote.png" # Save with a new name
    plt.savefig(filename)
    plt.close()

    model_images_smote[name] = filename

# ---------------------------
# 3. Explanation Function (VISUAL-BASED)
# (Re-using the same function, as its logic is general)
# ---------------------------
def explain_model_smote(name, df):
    row = df[df['Model'] == name].iloc[0]

    f1_score_val = round(row['F1-Score'], 3)
    accuracy_val = round(row['Accuracy'], 3)
    precision_val = round(row['Precision'], 3)
    recall_val = round(row['Recall'], 3)

    if f1_score_val > 0.7:
        performance = "good performance, showing a strong balance between precision and recall."
    elif f1_score_val > 0.4:
        performance = "moderate performance, indicating some ability to identify positive cases while maintaining reasonable precision."
    elif f1_score_val > 0.0:
        performance = "poor performance, struggling to effectively identify positive cases or maintain precision."
    else:
        performance = "very poor performance, essentially failing to identify any positive cases correctly (F1-Score of 0.0)."

    explanation = f"""
The {name} model achieved an F1-Score of {f1_score_val}, Accuracy of {accuracy_val}, Precision of {precision_val}, and Recall of {recall_val}.
This indicates {performance} The visualization shows how well actual vs. predicted labels align.
"""

    if name == "Logistic":
        explanation += " This linear model struggles with this dataset, suggesting features may not be linearly separable for effective classification."
    elif name == "Tree":
        explanation += " The decision tree's performance indicates difficulty in creating robust decision rules from the given features."
    elif name == "RF":
        explanation += " Despite the ensemble of trees, the Random Forest also struggles, pointing to fundamental limitations in feature informativeness or class imbalance issues."
    elif name == "GB":
        explanation += " Gradient Boosting's performance suggests that even complex, additive models find it hard to learn meaningful patterns, possibly due to small dataset size or inherent noise."
    elif name == "KNN":
        explanation += " The K-Nearest Neighbors model's performance suggests that local similarities are not strong enough to effectively classify the target variable."
    elif name == "SVC":
        explanation += " The Support Vector Classifier also struggles, indicating that a clear hyperplane to separate classes is not easily found in the feature space."
    elif name == "Ensemble":
        explanation += " Even by combining multiple models, the ensemble's performance is limited, highlighting the challenge of this classification task given the current data."
    elif name == "Bayesian":
        explanation += " The Gaussian Naive Bayes model assumes independence of features, and its performance suggests this assumption might not hold or that the features are not discriminative enough."

    return explanation

# ---------------------------
# 4. Table Styling (re-using existing function)
# ---------------------------
def create_table(df):
    data = [df.columns.tolist()] + df.values.tolist()
    table = Table(data)

    style = TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.grey),
        ('TEXTCOLOR',(0,0),(-1,0),colors.white),
        ('ALIGN',(0,0),(-1,-1),'CENTER'),
        ('FONTNAME', (0,0),(-1,0),'Helvetica-Bold'),
        ('GRID', (0,0), (-1,-1), 0.5, colors.black),
        ('LEFTPADDING', (0,0), (-1,-1), 6),
        ('RIGHTPADDING', (0,0), (-1,-1), 6),
        ('TOPPADDING', (0,0), (-1,-1), 3),
        ('BOTTOMPADDING', (0,0), (-1,-1), 3)
    ])

    table.setStyle(style)
    return table

# ---------------------------
# 5. Build PDF (updated for SMOTE results)
# ---------------------------
doc = SimpleDocTemplate("Final_Report_Styled_SMOTE.pdf", pagesize=letter) # New PDF name
styles = getSampleStyleSheet()
elements = []

# Title
elements.append(Paragraph("<b>Swine Health Prediction Model Report (After SMOTE)</b>", styles['Title']))
elements.append(Spacer(1,10))
elements.append(Paragraph(f"Generated on: {datetime.datetime.now()}", styles['Normal']))
elements.append(Spacer(1,20))

# Approach
elements.append(Paragraph("<b>1. Project Approach & SMOTE Application</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(Paragraph(
    "This project predicts swine health complications using merged datasets. "
    "A key focus was on building a robust machine learning pipeline that prevents data leakage "
    "and ensures fair model evaluation. Preprocessing included cleaning, encoding, and feature engineering, "
    "with the target variable 'Is_Swine_Influenza' created from aggregated symptom data for binary classification. "
    "To address the significant class imbalance, particularly in the training data, SMOTE (Synthetic Minority Over-sampling Technique) "
    "was applied to the training set to balance the classes. This section of the report reflects the model performance after SMOTE.",
    styles['Normal']))
elements.append(Spacer(1,20))

# Models
elements.append(Paragraph("<b>2. Models and Implementation</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(Paragraph(
    "The following classification models were implemented using scikit-learn: Logistic Regression, Decision Tree, "
    "Random Forest, Gradient Boosting, K-Nearest Neighbors, SVC, a dynamically selected Ensemble, and Gaussian Naive Bayes. "
    "All models were re-trained and re-evaluated on the SMOTE-resampled training data and original validation/test splits.",
    styles['Normal']))
elements.append(Spacer(1,20))

# Validation Table
elements.append(Paragraph("<b>3. Validation Results (After SMOTE)</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(create_table(val_df.round(3)))
elements.append(Spacer(1,20))

# Test Table
elements.append(Paragraph("<b>4. Test Results (After SMOTE)</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(create_table(test_df.round(3)))
elements.append(Spacer(1,20))

# Overall Graph
elements.append(Paragraph("<b>5. Model Comparison (After SMOTE)</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(Image("model_comparison_smote.png", width=400, height=200))
elements.append(Spacer(1,20))

# Best Model
elements.append(Paragraph("<b>6. Best Model on Test Set (After SMOTE)</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(Paragraph(
    f"The best performing model on the test set was the {best_model_test['Model']} model, with an F1-Score of {round(best_model_test['F1-Score'],3)} "
    f"(Accuracy={round(best_model_test['Accuracy'],3)}, Precision={round(best_model_test['Precision'],3)}, Recall={round(best_model_test['Recall'],3)}). "
    "It is important to note that despite SMOTE, the very small size of the test set and its continued severe class imbalance (only one class present in `y_test`) still lead to many metrics like F1-Score, Precision, and Recall defaulting to 0.0, and ROC-AUC remaining undefined. This significantly limits the interpretability of test results. SMOTE primarily addresses training set imbalance and cannot resolve issues stemming from a fundamentally imbalanced test set. Further data collection or a redefinition of the problem is crucial for more robust evaluation.",
    styles['Normal']))
elements.append(Spacer(1,20))

# Visual Analysis Section
elements.append(Paragraph("<b>7. Model Visual Analysis (After SMOTE)</b>", styles['Heading2']))
elements.append(Spacer(1,10))

for model_name in test_df['Model']:
    # Special handling for Bayesian if its plot isn't generated in the loop above
    if model_name == 'Bayesian' and 'Bayesian_plot_smote.png' not in model_images_smote:
        preds = bayes.predict(X_test)
        plt.figure()
        plt.scatter(y_test, preds, alpha=0.5)
        plt.xlabel("Actual (Is_Swine_Influenza)")
        plt.ylabel("Predicted (Is_Swine_Influenza)")
        plt.title("Bayesian Model Predictions (After SMOTE)")
        filename = "Bayesian_plot_smote.png"
        plt.savefig(filename)
        plt.close()
        model_images_smote[model_name] = filename

    if model_name in model_images_smote:
        elements.append(Paragraph(f"<b>{model_name} Model (After SMOTE)</b>", styles['Heading3']))
        elements.append(Spacer(1,8))
        elements.append(Image(model_images_smote[model_name], width=400, height=250))
        elements.append(Spacer(1,8))

        explanation = explain_model_smote(model_name, test_df) # Use updated explanation function if available
        elements.append(Paragraph(explanation, styles['Normal']))
        elements.append(Spacer(1,15))

# Reflection
elements.append(Paragraph("<b>8. Reflection and Improvements (After SMOTE)</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(Paragraph(
    "This report reflects the impact of applying SMOTE to the training data to address class imbalance. "
    "While SMOTE balanced the training set, the models' performance on the test set continues to be low, "
    "with metrics for the minority class often remaining at 0.0 or undefined. This is primarily because the "
    "test set itself contains only one class, making it impossible to evaluate a binary classifier meaningfully "
    "for the positive class. This situation underscores that resampling techniques like SMOTE are effective "
    "for improving training but cannot compensate for a severely imbalanced or single-class test set. "
    "The fundamental challenge remains the extremely limited and imbalanced nature of the dataset. "
    "Future work critically requires collecting more diverse and balanced data, or reconsidering the problem "
    "definition with a target variable that has sufficient examples in both classes across all splits. "
    "Without a representative test set, robust evaluation of classification models remains impossible.",
    styles['Normal']))

# Build PDF
doc.build(elements)

print("✅ Final_Report_Styled_SMOTE.pdf generated successfully")

### Investigating Feature Data

In [ ]:
print("First 5 rows of X_train:")
display(X_train.head())

print("\nNumerical features descriptive statistics:")
display(X_train[numerical_cols].describe())

print("\nKey observation regarding H1N1 summary features (global context):")
print("The H1N1 summary features ('c_a_s_e___c_o_u_n_t', 'C_u_m_u_l_a_t_i_v_e_ _n_o_._ _o_f_ _d_e_a_t_h_s') are global averages/maximums and therefore constant across all individual swine records. While they provide context, they cannot differentiate between individual swine cases and thus have zero predictive power for `Compounded_Complications` on their own. This further contributes to the poor model performance.")

print("\nThis analysis confirms that the target variable's nature and the non-discriminatory H1N1 global features are major contributors to the consistently negative R-squared values. The models are struggling to find patterns in data where the target is not truly continuous and some 'features' are constant.")

Step 13 Simple Visualization

In [ ]:
# ---------------------------
# Install Libraries
# ---------------------------
# These should ideally be at the very top of the notebook or handled once.
# Keeping here for self-contained cell, but noted for overall structure.
!pip install reportlab matplotlib

# ---------------------------
# Imports
# ---------------------------
import matplotlib.pyplot as plt
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.styles import getSampleStyleSheet
import datetime

# Ensure val_df and test_df are available from previous executed cells
# Assuming val_df, test_df, best_model_test are defined globally from previous steps

# ---------------------------
# 1. Create Overall Model Comparison Graph
# ---------------------------
plt.figure(figsize=(10,5))
# Changed from 'R2' to 'F1-Score' for classification
plt.bar(test_df['Model'], test_df['F1-Score'])
plt.title("Model Comparison (F1-Score)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("model_comparison.png")
plt.close()

# ---------------------------
# 2. Create Individual Model Plots
# ---------------------------
model_images = {}

# Note: Individual plots for classification (Actual vs. Predicted) are still relevant.
# For classification, a scatter plot might show clusters, or we could use confusion matrices.
# For now, keeping scatter to visualize how predictions align with actual labels.
for name, model in results.items(): # results from previous training cell
    preds = model.predict(X_test)

    plt.figure()
    plt.scatter(y_test, preds, alpha=0.5)
    plt.xlabel("Actual (Is_Swine_Influenza)")
    plt.ylabel("Predicted (Is_Swine_Influenza)")
    plt.title(f"{name} Model Predictions")

    filename = f"{name}_plot.png"
    plt.savefig(filename)
    plt.close()

    model_images[name] = filename

# ---------------------------
# 3. Explanation Function (VISUAL-BASED)
# ---------------------------
def explain_model(name, df):
    row = df[df['Model'] == name].iloc[0]

    # Changed from R2 and MAE to F1-Score and Accuracy for classification
    f1_score_val = round(row['F1-Score'], 3)
    accuracy_val = round(row['Accuracy'], 3)
    precision_val = round(row['Precision'], 3)
    recall_val = round(row['Recall'], 3)

    # Revised performance description based on F1-Score
    if f1_score_val > 0.7:
        performance = "good performance, showing a strong balance between precision and recall."
    elif f1_score_val > 0.4:
        performance = "moderate performance, indicating some ability to identify positive cases while maintaining reasonable precision."
    elif f1_score_val > 0.0:
        performance = "poor performance, struggling to effectively identify positive cases or maintain precision."
    else:
        performance = "very poor performance, essentially failing to identify any positive cases correctly (F1-Score of 0.0)."

    explanation = f"""
The {name} model achieved an F1-Score of {f1_score_val}, Accuracy of {accuracy_val}, Precision of {precision_val}, and Recall of {recall_val}.
This indicates {performance} The visualization shows how well actual vs. predicted labels align.
"""

    if name == "Logistic":
        explanation += " This linear model struggles with this dataset, suggesting features may not be linearly separable for effective classification."
    elif name == "Tree":
        explanation += " The decision tree's performance indicates difficulty in creating robust decision rules from the given features."
    elif name == "RF":
        explanation += " Despite the ensemble of trees, the Random Forest also struggles, pointing to fundamental limitations in feature informativeness or class imbalance issues."
    elif name == "GB":
        explanation += " Gradient Boosting's performance suggests that even complex, additive models find it hard to learn meaningful patterns, possibly due to small dataset size or inherent noise."
    elif name == "KNN":
        explanation += " The K-Nearest Neighbors model's performance suggests that local similarities are not strong enough to effectively classify the target variable."
    elif name == "SVC":
        explanation += " The Support Vector Classifier also struggles, indicating that a clear hyperplane to separate classes is not easily found in the feature space."
    elif name == "Ensemble":
        explanation += " Even by combining multiple models, the ensemble's performance is limited, highlighting the challenge of this classification task given the current data."
    elif name == "Bayesian":
        explanation += " The Gaussian Naive Bayes model assumes independence of features, and its performance suggests this assumption might not hold or that the features are not discriminative enough."

    return explanation

# ---------------------------
# 4. Table Styling
# ---------------------------
def create_table(df):
    data = [df.columns.tolist()] + df.values.tolist()
    table = Table(data)

    style = TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.grey),
        ('TEXTCOLOR',(0,0),(-1,0),colors.white),
        ('ALIGN',(0,0),(-1,-1),'CENTER'),
        ('FONTNAME', (0,0),(-1,0),'Helvetica-Bold'),
        ('GRID', (0,0), (-1,-1), 0.5, colors.black),
        ('LEFTPADDING', (0,0), (-1,-1), 6),
        ('RIGHTPADDING', (0,0), (-1,-1), 6),
        ('TOPPADDING', (0,0), (-1,-1), 3),
        ('BOTTOMPADDING', (0,0), (-1,-1), 3)
    ])

    table.setStyle(style)
    return table

# ---------------------------
# 5. Build PDF
# ---------------------------
doc = SimpleDocTemplate("Final_Report_Styled.pdf", pagesize=letter)
styles = getSampleStyleSheet()
elements = []

# Title
elements.append(Paragraph("<b>Swine Health Prediction Model Report</b>", styles['Title']))
elements.append(Spacer(1,10))
elements.append(Paragraph(f"Generated on: {datetime.datetime.now()}", styles['Normal']))
elements.append(Spacer(1,20))

# Approach
elements.append(Paragraph("<b>1. Project Approach</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(Paragraph(
    "This project predicts swine health complications using merged datasets. "
    "A key focus was on building a robust machine learning pipeline that prevents data leakage "
    "and ensures fair model evaluation. Preprocessing included cleaning, encoding, and feature engineering, "
    "with the target variable 'Is_Swine_Influenza' created from aggregated symptom data for binary classification.", # Updated target variable name
    styles['Normal']))
elements.append(Spacer(1,20))

# Models
elements.append(Paragraph("<b>2. Models and Implementation</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(Paragraph(
    "The following classification models were implemented using scikit-learn: Logistic Regression, Decision Tree, "
    "Random Forest, Gradient Boosting, K-Nearest Neighbors, SVC, a dynamically selected Ensemble, and Gaussian Naive Bayes. "
    "All models were trained and evaluated on preprocessed data after careful train/validation/test splits.", # Updated model list and classification focus
    styles['Normal']))
elements.append(Spacer(1,20))

# Validation Table
elements.append(Paragraph("<b>3. Validation Results</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(create_table(val_df.round(3))) # Use the updated val_df
elements.append(Spacer(1,20))

# Test Table
elements.append(Paragraph("<b>4. Test Results</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(create_table(test_df.round(3))) # Use the updated test_df
elements.append(Spacer(1,20))

# Overall Graph
elements.append(Paragraph("<b>5. Model Comparison</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(Image("model_comparison.png", width=400, height=200))
elements.append(Spacer(1,20))

# Best Model
elements.append(Paragraph("<b>6. Best Model on Test Set</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(Paragraph(
    f"The best performing model on the test set was the {best_model_test['Model']} model, with an F1-Score of {round(best_model_test['F1-Score'],3)} "
    f"(Accuracy={round(best_model_test['Accuracy'],3)}, Precision={round(best_model_test['Precision'],3)}, Recall={round(best_model_test['Recall'],3)}). "
    "It is important to note that due to the very small size of the test set and severe class imbalance (only one class present in `y_test`), many metrics like F1-Score, Precision, and Recall default to 0.0, and ROC-AUC is undefined. This significantly limits the interpretability of test results. Further data collection or resampling techniques would be crucial for a more robust evaluation.", # Updated metrics and added context about small test set
    styles['Normal']))
elements.append(Spacer(1,20))

# Visual Analysis Section
elements.append(Paragraph("<b>7. Model Visual Analysis</b>", styles['Heading2']))
elements.append(Spacer(1,10))

for model_name in test_df['Model']:
    if model_name not in model_images:
        # Special handling for Bayesian if its plot isn't generated in the loop above
        if model_name == 'Bayesian':
            # Create a simple plot for Bayesian if needed or skip
            preds = bayes.predict(X_test)
            plt.figure()
            plt.scatter(y_test, preds, alpha=0.5)
            plt.xlabel("Actual (Is_Swine_Influenza)")
            plt.ylabel("Predicted (Is_Swine_Influenza)")
            plt.title("Bayesian Model Predictions")
            filename = "Bayesian_plot.png"
            plt.savefig(filename)
            plt.close()
            model_images[model_name] = filename
        else:
            continue

    elements.append(Paragraph(f"<b>{model_name} Model</b>", styles['Heading3']))
    elements.append(Spacer(1,8))
    elements.append(Image(model_images[model_name], width=400, height=250))
    elements.append(Spacer(1,8))

    explanation = explain_model(model_name, test_df)
    elements.append(Paragraph(explanation, styles['Normal']))
    elements.append(Spacer(1,15))

# Reflection
elements.append(Paragraph("<b>8. Reflection and Improvements</b>", styles['Heading2']))
elements.append(Spacer(1,10))
elements.append(Paragraph(
    "This project significantly refactored the data pipeline to address critical issues such as data leakage, "
    "improper train/test split order, and artificial data explosion from cross-joins. Key improvements include: "
    "(1) Performing the train/validation/test split immediately after defining X and y, before any feature engineering. "
    "(2) Implementing one-hot encoding and outlier clipping *after* the data split, fitting these transformations only on the training set. "
    "(3) Replacing the problematic cross-join with a meaningful aggregation of H1N1 global context features. "
    "(4) Dynamically selecting ensemble models based on validation performance for improved reproducibility and justification. "
    "(5) Ensuring consistent evaluation of all models, including the Bayesian model, on the test set. "
    "However, despite these robust pipeline improvements, the models consistently exhibited very low F1-Scores and other classification metrics. "
    "Further investigation revealed that the test set `y_test` contained only one class, making many standard classification metrics (Precision, Recall, F1-Score for the positive class, and ROC-AUC) undefined or artificially zero. This severe class imbalance and small test set size mean that the models' true performance is not accurately reflected by the current test metrics. "
    "Additionally, the H1N1 global context features, while providing some general information, were constant across all individual swine records, offering no discriminatory power for predicting individual outcomes. "
    "These factors strongly indicate a fundamental mismatch between the chosen target variable, available features, and the small, imbalanced dataset, leading to the observed model challenges. "
    "Future work should prioritize collecting more balanced data, employing advanced resampling techniques, or redefining the problem with a more diverse target variable.", # Updated reflection to classification metrics and emphasized test set issues
    styles['Normal']))

# Build PDF
doc.build(elements)

print("✅ Final_Report_Styled.pdf generated successfully")

Step 14 Auto Generate Report